# NOTS-NIDS: Topology Demo
Visual demonstration of persistent homology on network traffic point clouds.
Shows how benign and attack traffic produce different persistence diagrams.

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.chdir('..')

import numpy as np
import matplotlib.pyplot as plt
from ripser import ripser
from config import Config
cfg = Config()

In [ ]:
# Generate synthetic benign and attack point clouds
np.random.seed(42)

# Benign: single Gaussian cluster
benign_cloud = np.random.randn(200, 5) * 0.3 + 0.5

# Attack: multi-modal (simulates coordinated attack pattern)
attack_cloud = np.vstack([
    np.random.randn(80, 5) * 0.2 + np.array([0.2, 0.8, 0.1, 0.9, 0.5]),
    np.random.randn(80, 5) * 0.2 + np.array([0.8, 0.1, 0.9, 0.2, 0.5]),
    np.random.randn(40, 5) * 0.15 + np.array([0.5, 0.5, 0.5, 0.5, 0.5]),
])

print(f'Benign cloud: {benign_cloud.shape}')
print(f'Attack cloud: {attack_cloud.shape}')

In [ ]:
# Compute persistence diagrams
from topology.ripser_filtration import compute_persistence_diagram, compute_betti_numbers

dgm_benign = compute_persistence_diagram(benign_cloud, max_dim=1, max_edge=2.0)
dgm_attack = compute_persistence_diagram(attack_cloud, max_dim=1, max_edge=2.0)

print(f'Benign: H0={len(dgm_benign["dgm_0"])} features, H1={len(dgm_benign["dgm_1"])} features')
print(f'Attack: H0={len(dgm_attack["dgm_0"])} features, H1={len(dgm_attack["dgm_1"])} features')

In [ ]:
# Visualise persistence diagrams side by side
from evaluation.plotter import plot_persistence_diagrams
plot_persistence_diagrams(
    dgm_benign, dgm_benign, dgm_attack,
    save_path='results/figures/topology_demo'
)

# Also display inline
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, dgm, title in zip(axes, [dgm_benign, dgm_attack], ['Benign', 'Attack']):
    ax.plot([0, 2.5], [0, 2.5], 'k--', alpha=0.3)
    h0 = dgm['dgm_0']
    h1 = dgm['dgm_1']
    if h0.size > 0:
        ax.scatter(h0[:, 0], h0[:, 1], c='blue', marker='o', s=20, alpha=0.6, label='H₀')
    if h1.size > 0:
        ax.scatter(h1[:, 0], h1[:, 1], c='red', marker='^', s=25, alpha=0.6, label='H₁')
    ax.set_title(f'{title} Traffic')
    ax.set_xlabel('Birth')
    ax.set_ylabel('Death')
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Compute Wasserstein distance
from topology.wasserstein import wasserstein_distance

W_benign_benign = wasserstein_distance(dgm_benign, dgm_benign)
W_benign_attack = wasserstein_distance(dgm_benign, dgm_attack)

print(f'W(benign, benign) = {W_benign_benign:.4f}  (should be ~0)')
print(f'W(benign, attack) = {W_benign_attack:.4f}  (should be large)')
print(f'\nThe attack traffic has a topological signature {W_benign_attack/max(W_benign_benign,0.01):.1f}x further from baseline.')

In [ ]:
# Betti numbers at varying filtration scales
epsilons = np.linspace(0, 2.0, 50)
betti_benign = [compute_betti_numbers(dgm_benign, e) for e in epsilons]
betti_attack = [compute_betti_numbers(dgm_attack, e) for e in epsilons]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, (b_name, bettis) in enumerate([('Benign', betti_benign), ('Attack', betti_attack)]):
    b0 = [b[0] for b in bettis]
    b1 = [b[1] for b in bettis]
    axes[i].plot(epsilons, b0, label='β₀ (components)', color='blue')
    axes[i].plot(epsilons, b1, label='β₁ (loops)', color='red')
    axes[i].set_title(f'{b_name} Traffic — Betti Numbers')
    axes[i].set_xlabel('Filtration Scale ε')
    axes[i].set_ylabel('Betti Number')
    axes[i].legend()
plt.tight_layout()
plt.show()